In [ ]:
# ============================================================
# Dataset: RealWaste (9 classes)




import os
import io
import time
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Config:
    num_classes = 9
    img_size = 224
    batch_size = 32
    num_workers = 4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    data_root = "/kaggle/input/datasets/nanda9949/9classes-waste/realwaste-main/RealWaste"

    # Teacher
    teacher_epochs = 25
    teacher_lr = 1e-4
    teacher_path = "teacher_effnetv2s_realwaste.pth"

    # Student (Slim MHA + distillation)
    student_epochs = 25
    student_lr = 1e-4
    temperature = 4.0
    alpha = 0.7
    mha_reduced_channels = 320          # 1280 → 320
    mha_heads = 4
    student_path = "student_cr_mhsa_fp32_realwaste.pth"
    weight_decay = 1e-4

    force_retrain = False              

cfg = Config()

# ============================================================
# 2. Data
# ============================================================
class TransformDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

def build_data():
    data_dir = Path(cfg.data_root)
    print("Dataset path:", data_dir.resolve())

    if not data_dir.exists():
        raise FileNotFoundError(
            f"Dataset folder not found: {data_dir}\n"
            "Please update cfg.data_root to the correct path."
        )

    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])

    train_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        norm,
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        norm,
    ])

    full = ImageFolder(root=data_dir, transform=None)
    cfg.class_names = full.classes
    print("Classes:", cfg.class_names)
    print("Total images:", len(full))

    n = len(full)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    n_test  = n - n_train - n_val

    train_ds, val_ds, test_ds = random_split(
        full, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

    train_set = TransformDataset(train_ds, train_tf)
    val_set   = TransformDataset(val_ds,   eval_tf)
    test_set  = TransformDataset(test_ds,  eval_tf)

    # Weighted sampler for class imbalance
    train_labels = [full.targets[i] for i in train_ds.indices]
    counts = Counter(train_labels)
    w = 1.0 / torch.tensor([counts[i] for i in range(cfg.num_classes)], dtype=torch.float)
    sample_w = torch.tensor([w[l] for l in train_labels])
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=cfg.num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
    return train_loader, val_loader, test_loader

class MultiHeadAttentionFusion(nn.Module):
    def __init__(self, in_channels, num_heads=4):
        super().__init__()
        assert in_channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.query = nn.Conv2d(in_channels, in_channels, 1)
        self.key   = nn.Conv2d(in_channels, in_channels, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.out_proj = nn.Conv2d(in_channels, in_channels, 1)
        self.norm = nn.BatchNorm2d(in_channels)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        q = self.query(x).view(B, self.num_heads, self.head_dim, H * W)
        k = self.key(x).view(B, self.num_heads, self.head_dim, H * W)
        v = self.value(x).view(B, self.num_heads, self.head_dim, H * W)
        attn = torch.einsum('bhdi,bhdj->bhij', q, k) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bhdj->bhdi', attn, v).reshape(B, C, H, W)
        out = self.norm(self.out_proj(out))
        return x + self.gamma * out

class SlimMHAFusion(nn.Module):
    def __init__(self, in_channels=1280, reduced=320, num_heads=4):
        super().__init__()
        assert reduced % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = reduced // num_heads
        self.reduced = reduced
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, reduced, 1, bias=False),
            nn.BatchNorm2d(reduced),
        )
        self.query = nn.Conv2d(reduced, reduced, 1)
        self.key   = nn.Conv2d(reduced, reduced, 1)
        self.value = nn.Conv2d(reduced, reduced, 1)
        self.expand = nn.Sequential(
            nn.Conv2d(reduced, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        r = self.reduce(x)
        q = self.query(r).view(B, self.num_heads, self.head_dim, H * W)
        k = self.key(r).view(B, self.num_heads, self.head_dim, H * W)
        v = self.value(r).view(B, self.num_heads, self.head_dim, H * W)
        attn = torch.einsum('bhdi,bhdj->bhij', q, k) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bhdj->bhdi', attn, v).reshape(B, self.reduced, H, W)
        out = self.expand(out)
        return x + self.gamma * out

class EfficientNetB0_MHA(nn.Module):
    def __init__(self, num_classes=9, pretrained=True, reduced=None, num_heads=4):
        super().__init__()
        backbone = models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None)
        self.features = backbone.features
        if reduced is None:
            self.mha = MultiHeadAttentionFusion(1280, num_heads)
        else:
            self.mha = SlimMHAFusion(1280, reduced, num_heads)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.mha(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def create_teacher(num_classes=9):
    model = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    distill = F.kl_div(soft_student, soft_teacher, reduction="batchmean") * (temperature ** 2)
    hard = F.cross_entropy(student_logits, labels)
    return alpha * distill + (1.0 - alpha) * hard

def train_one_epoch(model, loader, criterion, optimizer, device, teacher=None, temp=4.0, alpha=0.7):
    model.train()
    if teacher is not None:
        teacher.eval()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        if teacher is not None:
            with torch.no_grad():
                t_out = teacher(images)
            loss = distillation_loss(outputs, t_out, labels, temp, alpha)
        else:
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss.item(), acc=100.0 * correct / total)
    return running_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device, half=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    preds, labs = [], []
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images, labels = images.to(device), labels.to(device)
        if half:
            images = images.half()
        outputs = model(images).float()
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        pred = outputs.argmax(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
        preds.extend(pred.cpu().numpy())
        labs.extend(labels.cpu().numpy())
    acc = 100.0 * correct / total
    f1 = f1_score(labs, preds, average="macro")
    return running_loss / total, acc, f1, preds, labs


def get_teacher(train_loader, val_loader, criterion):
    teacher = create_teacher(cfg.num_classes).to(cfg.device)
    total, trainable = count_parameters(teacher)
    print(f"Teacher parameters: {total/1e6:.2f}M (trainable: {trainable/1e6:.2f}M)")

    if os.path.exists(cfg.teacher_path) and not cfg.force_retrain:
        teacher.load_state_dict(torch.load(cfg.teacher_path, map_location=cfg.device))
        print("Loaded saved teacher:", cfg.teacher_path)
    else:
        print("\n" + "="*60 + "\nSTAGE 1: Training Teacher (EfficientNetV2-S)\n" + "="*60)
        opt = optim.AdamW(teacher.parameters(), lr=cfg.teacher_lr, weight_decay=cfg.weight_decay)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.teacher_epochs)
        best_acc, best_wts = 0.0, copy.deepcopy(teacher.state_dict())

        for ep in range(cfg.teacher_epochs):
            print(f"\nTeacher Epoch {ep+1}/{cfg.teacher_epochs}")
            tl, ta = train_one_epoch(teacher, train_loader, criterion, opt, cfg.device)
            vl, va, vf, _, _ = evaluate(teacher, val_loader, criterion, cfg.device)
            sched.step()
            print(f"Train {tl:.4f}/{ta:.2f}% | Val {vl:.4f}/{va:.2f}% F1 {vf:.4f}")
            if va > best_acc:
                best_acc = va
                best_wts = copy.deepcopy(teacher.state_dict())

        teacher.load_state_dict(best_wts)
        torch.save(teacher.state_dict(), cfg.teacher_path)
        print(f"Teacher best Val Acc: {best_acc:.2f}%")

    for p in teacher.parameters():
        p.requires_grad = False
    return teacher.eval()

def get_student(teacher, train_loader, val_loader, criterion):
    student = EfficientNetB0_MHA(
        cfg.num_classes, True,
        cfg.mha_reduced_channels, cfg.mha_heads
    ).to(cfg.device)

    total, trainable = count_parameters(student)
    print(f"Slim student parameters: {total/1e6:.2f}M (trainable: {trainable/1e6:.2f}M)")

    if os.path.exists(cfg.student_path) and not cfg.force_retrain:
        ckpt = torch.load(cfg.student_path, map_location=cfg.device)
        student.load_state_dict(ckpt["model_state_dict"])
        print("Loaded saved slim student:", cfg.student_path)
        return student.eval()

    print("\n" + "="*60 + "\nSTAGE 2: Training Slim Student with Distillation\n" + "="*60)
    opt = optim.AdamW(student.parameters(), lr=cfg.student_lr, weight_decay=cfg.weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.student_epochs)
    best_acc, best_wts = 0.0, copy.deepcopy(student.state_dict())

    for ep in range(cfg.student_epochs):
        print(f"\nStudent Epoch {ep+1}/{cfg.student_epochs}")
        tl, ta = train_one_epoch(student, train_loader, criterion, opt, cfg.device,
                                 teacher=teacher, temp=cfg.temperature, alpha=cfg.alpha)
        vl, va, vf, _, _ = evaluate(student, val_loader, criterion, cfg.device)
        sched.step()
        print(f"Train {tl:.4f}/{ta:.2f}% | Val {vl:.4f}/{va:.2f}% F1 {vf:.4f}")
        if va > best_acc:
            best_acc = va
            best_wts = copy.deepcopy(student.state_dict())
            torch.save({
                "model_state_dict": best_wts,
                "class_names": cfg.class_names,
                "best_acc": best_acc
            }, cfg.student_path)
            print(f"→ Best student saved. Val Acc: {best_acc:.2f}%")

    student.load_state_dict(best_wts)
    print(f"Student best Val Acc: {best_acc:.2f}%")
    return student.eval()

def model_size_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6
    
def main():
    set_seed(42)
    print(f"Using device: {cfg.device}")
    print(f"Number of classes: {cfg.num_classes}")
    print(f"Dataset root: {cfg.data_root}")

    # Start CO2 tracker
    tracker = None
    if CO2_AVAILABLE:
        tracker = EmissionsTracker(project_name="SlimMHA_RealWaste", output_dir=".")
        tracker.start()
        print("CO₂ tracking started...")

    train_loader, val_loader, test_loader = build_data()
    criterion = nn.CrossEntropyLoss()

    results = []

    def log(name, model, acc, f1, note=""):
        size = model_size_mb(model)
        row = {
            "Model": name,
            "Size (MB)": round(size, 2),
            "Test Acc (%)": round(acc, 2),
            "Macro F1": round(f1, 4),
            "Note": note,
        }
        results.append(row)
        print(row)

    # ---------- Teacher ----------
    teacher = get_teacher(train_loader, val_loader, criterion)

    # ---------- Student ----------
    student = get_student(teacher, train_loader, val_loader, criterion)

    # ---------- Evaluation ----------
    print("\n" + "="*60 + "\nFINAL EVALUATION\n" + "="*60)

    # FP32
    _, acc32, f1_32, preds, labels = evaluate(student, test_loader, criterion, cfg.device)
    log("Slim-MHA FP32", student.cpu().eval(), acc32, f1_32)
    student.to(cfg.device)

    # FP16
    if cfg.device.type == "cuda":
        fp16 = copy.deepcopy(student).half().to(cfg.device).eval()
        _, acc16, f1_16, _, _ = evaluate(fp16, test_loader, criterion, cfg.device, half=True)
        log("Slim-MHA FP16", fp16.cpu(), acc16, f1_16, note="GPU inference")
        del fp16

    # ---------- Summary ----------
    df = pd.DataFrame(results)
    print("\n" + "="*60 + "\nRESULTS SUMMARY\n" + "="*60)
    print(df.to_string(index=False))
    df.to_csv("results_summary_realwaste.csv", index=False)

    # Per-class report
    print("\nClassification Report (FP32):")
    print(classification_report(labels, preds, target_names=cfg.class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(11, 9))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cfg.class_names, yticklabels=cfg.class_names)
    plt.title("Confusion Matrix - Slim-MHA FP32 (RealWaste 9 classes)")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("confusion_matrix_realwaste.png", dpi=150)
    plt.show()

    # ---------- CO2 ----------
    if tracker is not None:
        emissions = tracker.stop()
        print("\n" + "="*60)
        print(f"Total CO₂ emissions: {emissions:.6f} kg")
        print("="*60)
    else:
        print("\nCO₂ tracking was not available.")

    print("\nSaved files:")
    print("  - teacher_effnetv2s_realwaste.pth")
    print("  - student_cr_mhsa_fp32_realwaste.pth")
    print("  - results_summary_realwaste.csv")
    print("  - confusion_matrix_realwaste.png")

if __name__ == "__main__":
    main()

In [ ]:

# Dataset: sumn2u/garbage-classification-v2  (10 classes)


import os
import io
import time
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import kagglehub


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Config:
    num_classes = 10
    img_size = 224
    batch_size = 32
    num_workers = 4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Teacher
    teacher_epochs = 25
    teacher_lr = 1e-4
    teacher_path = "teacher_effnetv2s.pth"

    # Student
    student_epochs = 25
    student_lr = 1e-4
    temperature = 4.0
    alpha = 0.7
    mha_reduced_channels = 320
    mha_heads = 4
    student_path = "student_slim_mha_fp32.pth"
    weight_decay = 1e-4

    force_retrain = True

cfg = Config()

class TransformDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

def build_data():
    path = kagglehub.dataset_download("sumn2u/garbage-classification-v2")
    data_dir = Path(path) / "standardized_256"
    print("Dataset path:", data_dir)

    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])

    train_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        norm,
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        norm,
    ])

    full = ImageFolder(root=data_dir, transform=None)
    cfg.class_names = full.classes
    print("Classes:", cfg.class_names)
    print("Total images:", len(full))

    n = len(full)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    n_test  = n - n_train - n_val

    train_ds, val_ds, test_ds = random_split(
        full, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

    train_set = TransformDataset(train_ds, train_tf)
    val_set   = TransformDataset(val_ds,   eval_tf)
    test_set  = TransformDataset(test_ds,  eval_tf)

    train_labels = [full.targets[i] for i in train_ds.indices]
    counts = Counter(train_labels)
    w = 1.0 / torch.tensor([counts[i] for i in range(cfg.num_classes)], dtype=torch.float)
    sample_w = torch.tensor([w[l] for l in train_labels])
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=cfg.num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
    return train_loader, val_loader, test_loader

class MultiHeadAttentionFusion(nn.Module):
    def __init__(self, in_channels, num_heads=4):
        super().__init__()
        assert in_channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.query = nn.Conv2d(in_channels, in_channels, 1)
        self.key   = nn.Conv2d(in_channels, in_channels, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.out_proj = nn.Conv2d(in_channels, in_channels, 1)
        self.norm = nn.BatchNorm2d(in_channels)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        q = self.query(x).view(B, self.num_heads, self.head_dim, H * W)
        k = self.key(x).view(B, self.num_heads, self.head_dim, H * W)
        v = self.value(x).view(B, self.num_heads, self.head_dim, H * W)
        attn = torch.einsum('bhdi,bhdj->bhij', q, k) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bhdj->bhdi', attn, v).reshape(B, C, H, W)
        out = self.norm(self.out_proj(out))
        return x + self.gamma * out

class SlimMHAFusion(nn.Module):
    def __init__(self, in_channels=1280, reduced=320, num_heads=4):
        super().__init__()
        assert reduced % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = reduced // num_heads
        self.reduced = reduced
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, reduced, 1, bias=False),
            nn.BatchNorm2d(reduced),
        )
        self.query = nn.Conv2d(reduced, reduced, 1)
        self.key   = nn.Conv2d(reduced, reduced, 1)
        self.value = nn.Conv2d(reduced, reduced, 1)
        self.expand = nn.Sequential(
            nn.Conv2d(reduced, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        r = self.reduce(x)
        q = self.query(r).view(B, self.num_heads, self.head_dim, H * W)
        k = self.key(r).view(B, self.num_heads, self.head_dim, H * W)
        v = self.value(r).view(B, self.num_heads, self.head_dim, H * W)
        attn = torch.einsum('bhdi,bhdj->bhij', q, k) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bhdj->bhdi', attn, v).reshape(B, self.reduced, H, W)
        out = self.expand(out)
        return x + self.gamma * out

class EfficientNetB0_MHA(nn.Module):
    def __init__(self, num_classes=10, pretrained=True, reduced=None, num_heads=4):
        super().__init__()
        backbone = models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None)
        self.features = backbone.features
        if reduced is None:
            self.mha = MultiHeadAttentionFusion(1280, num_heads)
        else:
            self.mha = SlimMHAFusion(1280, reduced, num_heads)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(1280, num_classes))

    def forward(self, x):
        x = self.features(x)
        x = self.mha(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def create_teacher(num_classes=10):
    model = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(nn.Dropout(p=0.3), nn.Linear(in_features, num_classes))
    return model

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# ============================================================
# 4. Loss, train, eval
# ============================================================
def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    distill = F.kl_div(soft_student, soft_teacher, reduction="batchmean") * (temperature ** 2)
    hard = F.cross_entropy(student_logits, labels)
    return alpha * distill + (1.0 - alpha) * hard

def train_one_epoch(model, loader, criterion, optimizer, device, teacher=None, temp=4.0, alpha=0.7):
    model.train()
    if teacher is not None:
        teacher.eval()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        if teacher is not None:
            with torch.no_grad():
                t_out = teacher(images)
            loss = distillation_loss(outputs, t_out, labels, temp, alpha)
        else:
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss.item(), acc=100.0 * correct / total)
    return running_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device, half=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    preds, labs = [], []
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images, labels = images.to(device), labels.to(device)
        if half:
            images = images.half()
        outputs = model(images).float()
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        pred = outputs.argmax(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
        preds.extend(pred.cpu().numpy())
        labs.extend(labels.cpu().numpy())
    acc = 100.0 * correct / total
    f1 = f1_score(labs, preds, average="macro")
    return running_loss / total, acc, f1, preds, labs

def get_teacher(train_loader, val_loader, criterion):
    teacher = create_teacher(cfg.num_classes).to(cfg.device)
    total, trainable = count_parameters(teacher)
    print(f"Teacher parameters: {total/1e6:.2f}M (trainable: {trainable/1e6:.2f}M)")

    if os.path.exists(cfg.teacher_path) and not cfg.force_retrain:
        teacher.load_state_dict(torch.load(cfg.teacher_path, map_location=cfg.device))
        print("Loaded saved teacher:", cfg.teacher_path)
    else:
        print("\n" + "="*60 + "\nSTAGE 1: Training Teacher (EfficientNetV2-S)\n" + "="*60)
        opt = optim.AdamW(teacher.parameters(), lr=cfg.teacher_lr, weight_decay=cfg.weight_decay)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.teacher_epochs)
        best_acc, best_wts = 0.0, copy.deepcopy(teacher.state_dict())

        for ep in range(cfg.teacher_epochs):
            print(f"\nTeacher Epoch {ep+1}/{cfg.teacher_epochs}")
            tl, ta = train_one_epoch(teacher, train_loader, criterion, opt, cfg.device)
            vl, va, vf, _, _ = evaluate(teacher, val_loader, criterion, cfg.device)
            sched.step()
            print(f"Train {tl:.4f}/{ta:.2f}% | Val {vl:.4f}/{va:.2f}% F1 {vf:.4f}")
            if va > best_acc:
                best_acc = va
                best_wts = copy.deepcopy(teacher.state_dict())

        teacher.load_state_dict(best_wts)
        torch.save(teacher.state_dict(), cfg.teacher_path)
        print(f"Teacher best Val Acc: {best_acc:.2f}%")

    for p in teacher.parameters():
        p.requires_grad = False
    return teacher.eval()

def get_student(teacher, train_loader, val_loader, criterion):
    student = EfficientNetB0_MHA(
        cfg.num_classes, True,
        cfg.mha_reduced_channels, cfg.mha_heads
    ).to(cfg.device)

    total, trainable = count_parameters(student)
    print(f"Slim student parameters: {total/1e6:.2f}M (trainable: {trainable/1e6:.2f}M)")

    if os.path.exists(cfg.student_path) and not cfg.force_retrain:
        ckpt = torch.load(cfg.student_path, map_location=cfg.device)
        student.load_state_dict(ckpt["model_state_dict"])
        print("Loaded saved slim student:", cfg.student_path)
        return student.eval()

    print("\n" + "="*60 + "\nSTAGE 2: Training Slim Student with Distillation\n" + "="*60)
    opt = optim.AdamW(student.parameters(), lr=cfg.student_lr, weight_decay=cfg.weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.student_epochs)
    best_acc, best_wts = 0.0, copy.deepcopy(student.state_dict())

    for ep in range(cfg.student_epochs):
        print(f"\nStudent Epoch {ep+1}/{cfg.student_epochs}")
        tl, ta = train_one_epoch(student, train_loader, criterion, opt, cfg.device,
                                 teacher=teacher, temp=cfg.temperature, alpha=cfg.alpha)
        vl, va, vf, _, _ = evaluate(student, val_loader, criterion, cfg.device)
        sched.step()
        print(f"Train {tl:.4f}/{ta:.2f}% | Val {vl:.4f}/{va:.2f}% F1 {vf:.4f}")
        if va > best_acc:
            best_acc = va
            best_wts = copy.deepcopy(student.state_dict())
            torch.save({
                "model_state_dict": best_wts,
                "class_names": cfg.class_names,
                "best_acc": best_acc
            }, cfg.student_path)
            print(f"→ Best student saved. Val Acc: {best_acc:.2f}%")

    student.load_state_dict(best_wts)
    print(f"Student best Val Acc: {best_acc:.2f}%")
    return student.eval()


def model_size_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6

def main():
    set_seed(42)
    print(f"Using device: {cfg.device}")
    print(f"Number of classes: {cfg.num_classes}")

    emissions = {"Prepare": 0.0, "Develop": 0.0, "Deploy": 0.0}

    print("\n" + "="*60)
    print("STAGE: PREPARE (Dataset loading)")
    print("="*60)
    if CO2_AVAILABLE:
        tracker_prep = EmissionsTracker(project_name="SlimMHA_Prepare", output_dir=".", log_level="error")
        tracker_prep.start()

    train_loader, val_loader, test_loader = build_data()
    criterion = nn.CrossEntropyLoss()

    if CO2_AVAILABLE:
        emissions["Prepare"] = tracker_prep.stop()
        print(f"Prepare CO₂: {emissions['Prepare']:.6f} kg")


    print("\n" + "="*60)
    print("STAGE: DEVELOP (Training Teacher + Student)")
    print("="*60)
    if CO2_AVAILABLE:
        tracker_dev = EmissionsTracker(project_name="SlimMHA_Develop", output_dir=".", log_level="error")
        tracker_dev.start()

    teacher = get_teacher(train_loader, val_loader, criterion)
    student = get_student(teacher, train_loader, val_loader, criterion)

    if CO2_AVAILABLE:
        emissions["Develop"] = tracker_dev.stop()
        print(f"Develop CO₂: {emissions['Develop']:.6f} kg")

    # ==================== DEPLOY ====================
    print("\n" + "="*60)
    print("STAGE: DEPLOY (Evaluation / Inference)")
    print("="*60)
    if CO2_AVAILABLE:
        tracker_dep = EmissionsTracker(project_name="SlimMHA_Deploy", output_dir=".", log_level="error")
        tracker_dep.start()

    results = []

    def log(name, model, acc, f1, note=""):
        size = model_size_mb(model)
        row = {
            "Model": name,
            "Size (MB)": round(size, 2),
            "Test Acc (%)": round(acc, 2),
            "Macro F1": round(f1, 4),
            "Note": note,
        }
        results.append(row)
        print(row)

    # FP32
    _, acc32, f1_32, preds, labels = evaluate(student, test_loader, criterion, cfg.device)
    log("Slim-MHA FP32", student.cpu().eval(), acc32, f1_32)
    student.to(cfg.device)

    # FP16
    if cfg.device.type == "cuda":
        fp16 = copy.deepcopy(student).half().to(cfg.device).eval()
        _, acc16, f1_16, _, _ = evaluate(fp16, test_loader, criterion, cfg.device, half=True)
        log("Slim-MHA FP16", fp16.cpu(), acc16, f1_16, note="GPU inference")
        del fp16

    if CO2_AVAILABLE:
        emissions["Deploy"] = tracker_dep.stop()
        print(f"Deploy CO₂: {emissions['Deploy']:.6f} kg")

    # ==================== RESULTS ====================
    df = pd.DataFrame(results)
    print("\n" + "="*60)
    print("RESULTS SUMMARY")
    print("="*60)
    print(df.to_string(index=False))
    df.to_csv("results_summary.csv", index=False)

    print("\nClassification Report (FP32):")
    print(classification_report(labels, preds, target_names=cfg.class_names, digits=4))

    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cfg.class_names, yticklabels=cfg.class_names)
    plt.title("Confusion Matrix - Slim-MHA FP32")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("confusion_matrix_final.png", dpi=150)
    plt.show()
    print("\n" + "="*60)
    print("CO₂ EMISSIONS (kg)")
    print("="*60)
    print(f"{'Stage':<12} {'CO₂ (kg)':>12}")
    print("-" * 26)
    print(f"{'Prepare':<12} {emissions['Prepare']:>12.6f}")
    print(f"{'Develop':<12} {emissions['Develop']:>12.6f}")
    print(f"{'Deploy':<12} {emissions['Deploy']:>12.6f}")
    print("-" * 26)
    total_co2 = sum(emissions.values())
    print(f"{'Total':<12} {total_co2:>12.6f}")
    print("="*60)

    co2_df = pd.DataFrame([
        {"Stage": "Prepare", "CO2_kg": emissions["Prepare"]},
        {"Stage": "Develop", "CO2_kg": emissions["Develop"]},
        {"Stage": "Deploy",  "CO2_kg": emissions["Deploy"]},
        {"Stage": "Total",   "CO2_kg": total_co2},
    ])
    co2_df.to_csv("co2_emissions.csv", index=False)

    print("\nSaved files:")
    print("  - teacher_effnetv2s.pth")
    print("  - student_slim_mha_fp32.pth")
    print("  - results_summary.csv")
    print("  - co2_emissions.csv")
    print("  - confusion_matrix_final.png")

if __name__ == "__main__":
    main()

In [ ]:

# Dataset: mostafaabla/garbage-classification  (12 classes)


import os
import io
import time
import copy
import random
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split, WeightedRandomSampler
from torchvision import transforms, models
from torchvision.datasets import ImageFolder
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import kagglehub



def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

class Config:
    num_classes = 12
    img_size = 224
    batch_size = 32
    num_workers = 4
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Teacher
    teacher_epochs = 25
    teacher_lr = 1e-4
    teacher_path = "teacher_effnetv2s_12cls.pth"

    # Student (Slim MHA + distillation)
    student_epochs = 25
    student_lr = 1e-4
    temperature = 4.0
    alpha = 0.7
    mha_reduced_channels = 320          # 1280 → 320
    mha_heads = 4
    student_path = "student_cr_mhsa_fp32_12cls.pth"
    weight_decay = 1e-4

    force_retrain = False               # True = ignore saved checkpoints

cfg = Config()

class TransformDataset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform

    def __getitem__(self, idx):
        x, y = self.subset[idx]
        if self.transform:
            x = self.transform(x)
        return x, y

    def __len__(self):
        return len(self.subset)

def build_data():
    path = kagglehub.dataset_download("mostafaabla/garbage-classification")
    data_dir = Path(path)
    print("Dataset path:", data_dir)

    # Find the actual image root (handles nested folders)
    if (data_dir / "garbage_classification").exists():
        data_dir = data_dir / "garbage_classification"
    elif any((data_dir / d).is_dir() for d in data_dir.iterdir()):
        # already has class folders
        pass

    norm = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225])

    train_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        norm,
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((cfg.img_size, cfg.img_size)),
        transforms.ToTensor(),
        norm,
    ])

    full = ImageFolder(root=data_dir, transform=None)
    cfg.class_names = full.classes
    print("Classes:", cfg.class_names)
    print("Total images:", len(full))

    n = len(full)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    n_test  = n - n_train - n_val

    train_ds, val_ds, test_ds = random_split(
        full, [n_train, n_val, n_test],
        generator=torch.Generator().manual_seed(42)
    )

    train_set = TransformDataset(train_ds, train_tf)
    val_set   = TransformDataset(val_ds,   eval_tf)
    test_set  = TransformDataset(test_ds,  eval_tf)

    # Weighted sampler for class imbalance
    train_labels = [full.targets[i] for i in train_ds.indices]
    counts = Counter(train_labels)
    w = 1.0 / torch.tensor([counts[i] for i in range(cfg.num_classes)], dtype=torch.float)
    sample_w = torch.tensor([w[l] for l in train_labels])
    sampler = WeightedRandomSampler(sample_w, len(sample_w), replacement=True)

    pin = torch.cuda.is_available()
    train_loader = DataLoader(train_set, batch_size=cfg.batch_size, sampler=sampler,
                              num_workers=cfg.num_workers, pin_memory=pin)
    val_loader   = DataLoader(val_set, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)
    test_loader  = DataLoader(test_set, batch_size=cfg.batch_size, shuffle=False,
                              num_workers=cfg.num_workers, pin_memory=pin)

    print(f"Train: {len(train_set)} | Val: {len(val_set)} | Test: {len(test_set)}")
    return train_loader, val_loader, test_loader

# ============================================================
# 3. Models
# ============================================================
class MultiHeadAttentionFusion(nn.Module):
    def __init__(self, in_channels, num_heads=4):
        super().__init__()
        assert in_channels % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = in_channels // num_heads
        self.query = nn.Conv2d(in_channels, in_channels, 1)
        self.key   = nn.Conv2d(in_channels, in_channels, 1)
        self.value = nn.Conv2d(in_channels, in_channels, 1)
        self.out_proj = nn.Conv2d(in_channels, in_channels, 1)
        self.norm = nn.BatchNorm2d(in_channels)
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        q = self.query(x).view(B, self.num_heads, self.head_dim, H * W)
        k = self.key(x).view(B, self.num_heads, self.head_dim, H * W)
        v = self.value(x).view(B, self.num_heads, self.head_dim, H * W)
        attn = torch.einsum('bhdi,bhdj->bhij', q, k) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bhdj->bhdi', attn, v).reshape(B, C, H, W)
        out = self.norm(self.out_proj(out))
        return x + self.gamma * out

class SlimMHAFusion(nn.Module):
    def __init__(self, in_channels=1280, reduced=320, num_heads=4):
        super().__init__()
        assert reduced % num_heads == 0
        self.num_heads = num_heads
        self.head_dim = reduced // num_heads
        self.reduced = reduced
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, reduced, 1, bias=False),
            nn.BatchNorm2d(reduced),
        )
        self.query = nn.Conv2d(reduced, reduced, 1)
        self.key   = nn.Conv2d(reduced, reduced, 1)
        self.value = nn.Conv2d(reduced, reduced, 1)
        self.expand = nn.Sequential(
            nn.Conv2d(reduced, in_channels, 1, bias=False),
            nn.BatchNorm2d(in_channels),
        )
        self.gamma = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        B, C, H, W = x.shape
        r = self.reduce(x)
        q = self.query(r).view(B, self.num_heads, self.head_dim, H * W)
        k = self.key(r).view(B, self.num_heads, self.head_dim, H * W)
        v = self.value(r).view(B, self.num_heads, self.head_dim, H * W)
        attn = torch.einsum('bhdi,bhdj->bhij', q, k) * (self.head_dim ** -0.5)
        attn = F.softmax(attn, dim=-1)
        out = torch.einsum('bhij,bhdj->bhdi', attn, v).reshape(B, self.reduced, H, W)
        out = self.expand(out)
        return x + self.gamma * out

class EfficientNetB0_MHA(nn.Module):
    def __init__(self, num_classes=12, pretrained=True, reduced=None, num_heads=4):
        super().__init__()
        backbone = models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None)
        self.features = backbone.features
        if reduced is None:
            self.mha = MultiHeadAttentionFusion(1280, num_heads)
        else:
            self.mha = SlimMHAFusion(1280, reduced, num_heads)
        self.avgpool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.3),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.mha(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def create_teacher(num_classes=12):
    model = models.efficientnet_v2_s(weights="IMAGENET1K_V1")
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def distillation_loss(student_logits, teacher_logits, labels, temperature=4.0, alpha=0.7):
    soft_teacher = F.softmax(teacher_logits / temperature, dim=1)
    soft_student = F.log_softmax(student_logits / temperature, dim=1)
    distill = F.kl_div(soft_student, soft_teacher, reduction="batchmean") * (temperature ** 2)
    hard = F.cross_entropy(student_logits, labels)
    return alpha * distill + (1.0 - alpha) * hard

def train_one_epoch(model, loader, criterion, optimizer, device, teacher=None, temp=4.0, alpha=0.7):
    model.train()
    if teacher is not None:
        teacher.eval()
    running_loss, correct, total = 0.0, 0, 0
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        if teacher is not None:
            with torch.no_grad():
                t_out = teacher(images)
            loss = distillation_loss(outputs, t_out, labels, temp, alpha)
        else:
            loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        correct += outputs.argmax(1).eq(labels).sum().item()
        total += labels.size(0)
        pbar.set_postfix(loss=loss.item(), acc=100.0 * correct / total)
    return running_loss / total, 100.0 * correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device, half=False):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    preds, labs = [], []
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images, labels = images.to(device), labels.to(device)
        if half:
            images = images.half()
        outputs = model(images).float()
        loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        pred = outputs.argmax(1)
        correct += pred.eq(labels).sum().item()
        total += labels.size(0)
        preds.extend(pred.cpu().numpy())
        labs.extend(labels.cpu().numpy())
    acc = 100.0 * correct / total
    f1 = f1_score(labs, preds, average="macro")
    return running_loss / total, acc, f1, preds, labs

def get_teacher(train_loader, val_loader, criterion):
    teacher = create_teacher(cfg.num_classes).to(cfg.device)
    total, trainable = count_parameters(teacher)
    print(f"Teacher parameters: {total/1e6:.2f}M (trainable: {trainable/1e6:.2f}M)")

    if os.path.exists(cfg.teacher_path) and not cfg.force_retrain:
        teacher.load_state_dict(torch.load(cfg.teacher_path, map_location=cfg.device))
        print("Loaded saved teacher:", cfg.teacher_path)
    else:
        print("\n" + "="*60 + "\nSTAGE 1: Training Teacher (EfficientNetV2-S)\n" + "="*60)
        opt = optim.AdamW(teacher.parameters(), lr=cfg.teacher_lr, weight_decay=cfg.weight_decay)
        sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.teacher_epochs)
        best_acc, best_wts = 0.0, copy.deepcopy(teacher.state_dict())

        for ep in range(cfg.teacher_epochs):
            print(f"\nTeacher Epoch {ep+1}/{cfg.teacher_epochs}")
            tl, ta = train_one_epoch(teacher, train_loader, criterion, opt, cfg.device)
            vl, va, vf, _, _ = evaluate(teacher, val_loader, criterion, cfg.device)
            sched.step()
            print(f"Train {tl:.4f}/{ta:.2f}% | Val {vl:.4f}/{va:.2f}% F1 {vf:.4f}")
            if va > best_acc:
                best_acc = va
                best_wts = copy.deepcopy(teacher.state_dict())

        teacher.load_state_dict(best_wts)
        torch.save(teacher.state_dict(), cfg.teacher_path)
        print(f"Teacher best Val Acc: {best_acc:.2f}%")

    for p in teacher.parameters():
        p.requires_grad = False
    return teacher.eval()

def get_student(teacher, train_loader, val_loader, criterion):
    student = EfficientNetB0_MHA(
        cfg.num_classes, True,
        cfg.mha_reduced_channels, cfg.mha_heads
    ).to(cfg.device)

    total, trainable = count_parameters(student)
    print(f"Slim student parameters: {total/1e6:.2f}M (trainable: {trainable/1e6:.2f}M)")

    if os.path.exists(cfg.student_path) and not cfg.force_retrain:
        ckpt = torch.load(cfg.student_path, map_location=cfg.device)
        student.load_state_dict(ckpt["model_state_dict"])
        print("Loaded saved slim student:", cfg.student_path)
        return student.eval()

    print("\n" + "="*60 + "\nSTAGE 2: Training Slim Student with Distillation\n" + "="*60)
    opt = optim.AdamW(student.parameters(), lr=cfg.student_lr, weight_decay=cfg.weight_decay)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.student_epochs)
    best_acc, best_wts = 0.0, copy.deepcopy(student.state_dict())

    for ep in range(cfg.student_epochs):
        print(f"\nStudent Epoch {ep+1}/{cfg.student_epochs}")
        tl, ta = train_one_epoch(student, train_loader, criterion, opt, cfg.device,
                                 teacher=teacher, temp=cfg.temperature, alpha=cfg.alpha)
        vl, va, vf, _, _ = evaluate(student, val_loader, criterion, cfg.device)
        sched.step()
        print(f"Train {tl:.4f}/{ta:.2f}% | Val {vl:.4f}/{va:.2f}% F1 {vf:.4f}")
        if va > best_acc:
            best_acc = va
            best_wts = copy.deepcopy(student.state_dict())
            torch.save({
                "model_state_dict": best_wts,
                "class_names": cfg.class_names,
                "best_acc": best_acc
            }, cfg.student_path)
            print(f"→ Best student saved. Val Acc: {best_acc:.2f}%")

    student.load_state_dict(best_wts)
    print(f"Student best Val Acc: {best_acc:.2f}%")
    return student.eval()

def model_size_mb(m):
    buf = io.BytesIO()
    torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1e6



def main():
    set_seed(42)
    print(f"Using device: {cfg.device}")
    print(f"Number of classes: {cfg.num_classes}")

    # Start CO2 tracker
    tracker = None
    if CO2_AVAILABLE:
        tracker = EmissionsTracker(project_name="SlimMHA_12cls", output_dir=".")
        tracker.start()
        print("CO₂ tracking started...")

    train_loader, val_loader, test_loader = build_data()
    criterion = nn.CrossEntropyLoss()

    results = []

    def log(name, model, acc, f1, note=""):
        size = model_size_mb(model)
        row = {
            "Model": name,
            "Size (MB)": round(size, 2),
            "Test Acc (%)": round(acc, 2),
            "Macro F1": round(f1, 4),
            "Note": note,
        }
        results.append(row)
        print(row)

    # ---------- Teacher ----------
    teacher = get_teacher(train_loader, val_loader, criterion)

    # ---------- Student ----------
    student = get_student(teacher, train_loader, val_loader, criterion)

    # ---------- Evaluation ----------
    print("\n" + "="*60 + "\nFINAL EVALUATION\n" + "="*60)

    # FP32
    _, acc32, f1_32, preds, labels = evaluate(student, test_loader, criterion, cfg.device)
    log("Slim-MHA FP32", student.cpu().eval(), acc32, f1_32)
    student.to(cfg.device)

    # FP16
    if cfg.device.type == "cuda":
        fp16 = copy.deepcopy(student).half().to(cfg.device).eval()
        _, acc16, f1_16, _, _ = evaluate(fp16, test_loader, criterion, cfg.device, half=True)
        log("Slim-MHA FP16", fp16.cpu(), acc16, f1_16, note="GPU inference")
        del fp16

    # ---------- Summary ----------
    df = pd.DataFrame(results)
    print("\n" + "="*60 + "\nRESULTS SUMMARY\n" + "="*60)
    print(df.to_string(index=False))
    df.to_csv("results_summary_12cls.csv", index=False)

    # Per-class report
    print("\nClassification Report (FP32):")
    print(classification_report(labels, preds, target_names=cfg.class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(labels, preds)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=cfg.class_names, yticklabels=cfg.class_names)
    plt.title("Confusion Matrix - Slim-MHA FP32 (12 classes)")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.savefig("confusion_matrix_12cls.png", dpi=150)
    plt.show()

    # ---------- CO2 ----------
    if tracker is not None:
        emissions = tracker.stop()
        print("\n" + "="*60)
        print(f"Total CO₂ emissions: {emissions:.6f} kg")
        print("="*60)
    else:
        print("\nCO₂ tracking was not available.")

    print("\nSaved files:")
    print("  - teacher_effnetv2s_12cls.pth")
    print("  - student_cr_mhsa_fp32_12cls.pth")
    print("  - results_summary_12cls.csv")
    print("  - confusion_matrix_12cls.png")

if __name__ == "__main__":
    main()